# 2.14 Bayesian optimization

Bayesian optimization spends expensive evaluations where uncertainty and promise make the next observation most valuable.

This lesson optimizes when function calls are precious. A probabilistic surrogate turns stochastic thinking into an algorithm, and acquisition functions trade off exploitation and exploration.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from math import erf
from math import sqrt

SEED = 214
rng = np.random.default_rng(SEED)


def rbf_kernel(x_left, x_right, length_scale=0.35, variance=1.0):
    x_left = np.atleast_2d(x_left).astype(float)
    x_right = np.atleast_2d(x_right).astype(float)
    diffs = x_left[:, None, :] - x_right[None, :, :]
    squared_distance = np.sum(diffs * diffs, axis=2)
    return variance * np.exp(-0.5 * squared_distance / (length_scale * length_scale))


def gp_posterior(x_train, y_train, x_query, length_scale=0.35, noise=1e-6):
    x_train = np.atleast_2d(x_train).astype(float)
    x_query = np.atleast_2d(x_query).astype(float)
    y_train = np.asarray(y_train, dtype=float)
    kernel_train = rbf_kernel(x_train, x_train, length_scale=length_scale)
    kernel_train = kernel_train + noise * np.eye(len(x_train))
    kernel_cross = rbf_kernel(x_train, x_query, length_scale=length_scale)
    kernel_query = rbf_kernel(x_query, x_query, length_scale=length_scale)
    alpha = np.linalg.solve(kernel_train, y_train)
    mean = kernel_cross.T @ alpha
    solve_cross = np.linalg.solve(kernel_train, kernel_cross)
    covariance = kernel_query - kernel_cross.T @ solve_cross
    variance = np.maximum(np.diag(covariance), 1e-12)
    return mean, np.sqrt(variance)


def normal_pdf(z):
    return np.exp(-0.5 * z * z) / np.sqrt(2.0 * np.pi)


def normal_cdf(z):
    z = np.asarray(z, dtype=float)
    return 0.5 * (1.0 + np.vectorize(erf)(z / sqrt(2.0)))


def expected_improvement(mean, sigma, best_score):
    improvement = mean - best_score
    safe_sigma = np.maximum(sigma, 1e-12)
    z = improvement / safe_sigma
    return improvement * normal_cdf(z) + safe_sigma * normal_pdf(z)


def bayes_opt_step(x_train, y_train, candidates, length_scale=0.35, noise=1e-6, kappa=1.5):
    mean, sigma = gp_posterior(x_train, y_train, candidates, length_scale=length_scale, noise=noise)
    acquisition = mean + kappa * sigma
    next_index = int(np.argmax(acquisition))
    return candidates[next_index], mean, sigma, acquisition


def make_grid(bounds, points_per_axis):
    axes = []
    for low, high in bounds:
        axes.append(np.linspace(low, high, points_per_axis))
    mesh = np.meshgrid(*axes, indexing="xy")
    stacked = np.stack([axis.ravel() for axis in mesh], axis=1)
    return stacked


def sigmoid(scores):
    return 1.0 / (1.0 + np.exp(-np.clip(scores, -30, 30)))


def load_logistic_data(max_features=8):
    try:
        from sklearn.datasets import load_breast_cancer

        data = load_breast_cancer()
        features = data.data[:, :max_features].astype(float)
        target = data.target.astype(float)
    except Exception:
        features = rng.normal(size=(180, max_features))
        true_weights = np.linspace(-1.0, 1.0, max_features)
        target = (sigmoid(features @ true_weights) > 0.5).astype(float)

    split = int(0.7 * len(features))
    train_x = features[:split]
    valid_x = features[split:]
    train_y = target[:split]
    valid_y = target[split:]
    mean = train_x.mean(axis=0)
    scale = train_x.std(axis=0) + 1e-6
    train_x = (train_x - mean) / scale
    valid_x = (valid_x - mean) / scale
    return train_x, train_y, valid_x, valid_y


LOGISTIC_DATA = load_logistic_data(max_features=8)


def logistic_hyperparameter_loss(point):
    log_lr, log_l2 = point
    learning_rate = 10.0 ** log_lr
    l2 = 10.0 ** log_l2
    train_x, train_y, valid_x, valid_y = LOGISTIC_DATA
    weights = np.zeros(train_x.shape[1])
    bias = 0.0

    for step in range(80):
        prediction = sigmoid(train_x @ weights + bias)
        error = prediction - train_y
        gradient = train_x.T @ error / len(train_y) + l2 * weights
        bias_gradient = float(np.mean(error))
        weights = weights - learning_rate * gradient
        bias = bias - learning_rate * bias_gradient

    valid_prediction = sigmoid(valid_x @ weights + bias)
    loss = -np.mean(valid_y * np.log(valid_prediction + 1e-9) + (1.0 - valid_y) * np.log(1.0 - valid_prediction + 1e-9))
    return float(loss)


def bayesian_minimize(objective, bounds, iterations=16, initial_count=3, grid_size=19, length_scale=0.35, noise=1e-5, kappa=1.5):
    candidates = make_grid(bounds, grid_size)
    chosen = np.linspace(0, len(candidates) - 1, initial_count, dtype=int)
    x_train = candidates[chosen]
    losses = np.array([objective(x) for x in x_train], dtype=float)
    best_losses = [float(np.min(losses))]
    path = [x.copy() for x in x_train]

    for step in range(iterations):
        scores = -losses
        next_x, mean, sigma, acquisition = bayes_opt_step(x_train, scores, candidates, length_scale=length_scale, noise=noise, kappa=kappa)
        next_loss = float(objective(next_x))
        x_train = np.vstack([x_train, next_x])
        losses = np.append(losses, next_loss)
        best_losses.append(float(np.min(losses)))
        path.append(next_x.copy())

    return {
        "x": x_train,
        "losses": losses,
        "best_curve": np.array(best_losses),
        "path": np.array(path),
        "candidates": candidates,
    }


def build_bayes_ladder():
    def d1(point):
        x = point[0]
        return (x - 0.75) ** 2 + 0.02

    def d2(point):
        x, y = point
        return 0.02 * (x - 0.4) ** 2 + 3.0 * (y + 0.25) ** 2 + 0.05

    def d3(point):
        x, y = point
        bowl = 0.15 * (x - 0.2) ** 2 + 0.1 * (y + 0.4) ** 2
        waves = 0.12 * np.sin(5 * x) * np.cos(4 * y)
        return bowl + waves + 0.25

    def d5(point):
        penalty = 0.0
        if point[0] + point[1] > 1.0:
            penalty = 0.2 + 0.5 * (point[0] + point[1] - 1.0)
        center = np.array([0.2, 0.6, -0.4])
        return float(np.sum((point - center) ** 2) * 0.08 + 0.05 * np.cos(6 * point[2]) + penalty + 0.12)

    return [
        {"name": "D1 one-dimensional cheap black box", "objective": d1, "bounds": [(-1.0, 2.0)], "grid": 41},
        {"name": "D2 anisotropic two-dimensional black box", "objective": d2, "bounds": [(-1.0, 1.5), (-1.0, 1.0)], "grid": 21},
        {"name": "D3 multimodal response surface", "objective": d3, "bounds": [(-1.5, 1.5), (-1.5, 1.5)], "grid": 23},
        {"name": "D4 real logistic-regression validation loss", "objective": logistic_hyperparameter_loss, "bounds": [(-3.0, -0.2), (-5.0, -1.0)], "grid": 17},
        {"name": "D5 constrained high-dimensional search", "objective": d5, "bounds": [(0.0, 1.0), (0.0, 1.0), (-1.0, 1.0)], "grid": 13},
    ]


## The concept, built once: GP surrogate plus acquisition

Bayesian optimization keeps $D=\{(x_i,y_i)\}$, posterior mean $\mu(x)$, posterior standard deviation $\sigma(x)$, and an acquisition such as $a(x)=\mu(x)+\kappa\sigma(x)$ for maximization.

The lightweight Gaussian process below uses an RBF kernel and exact conditioning on a small candidate grid.

In [ ]:

x_obs = np.array([[0.0], [1.0], [2.0]])
y_obs = np.array([0.0, 1.0, 0.0])
query = np.array([[1.0]])

mean_at_one, sigma_at_one = gp_posterior(x_obs, y_obs, query, length_scale=0.25, noise=1e-8)
current_best = float(np.max(y_obs))

print("posterior mean at x=1", mean_at_one[0])
print("posterior sigma at x=1", sigma_at_one[0])
print("current best", current_best)

assert abs(mean_at_one[0] - 1.0) < 1e-3
assert current_best == 1.0


## Expected improvement and UCB use the lesson numbers

If the current best is $1.0$, a candidate with $\mu=0.8$ and $\sigma=0.3$ has $z=(0.8-1.0)/0.3=-0.6667$ and $EI\approx0.0453$. For UCB, $\mu=0.6$, $\sigma=0.2$, and $\kappa=2$ give $a=1.0$.

In [ ]:

ei_mean = np.array([0.8])
ei_sigma = np.array([0.3])
ei_value = expected_improvement(ei_mean, ei_sigma, best_score=1.0)[0]
ucb_low = 0.6 + 0.5 * 0.2
ucb_high = 0.6 + 2.0 * 0.2

print("EI", ei_value)
print("UCB kappa=0.5", ucb_low)
print("UCB kappa=2", ucb_high)

assert abs(ei_value - 0.0453) < 5e-4
assert abs(ucb_low - 0.7) < 1e-12
assert abs(ucb_high - 1.0) < 1e-12


## The dataset ladder

The rungs progress from a one-dimensional quadratic to anisotropic, multimodal, hyperparameter-like, and constrained search spaces. Every rung is small enough for CPU-only candidate grids.

In [ ]:

rungs = build_bayes_ladder()

for index, rung in enumerate(rungs, start=1):
    dimensions = len(rung["bounds"])
    candidates = make_grid(rung["bounds"], rung["grid"])
    sample_losses = [rung["objective"](candidate) for candidate in candidates[:3]]
    print(f"D{index}: {rung['name']}")
    print("  dimensions", dimensions, "candidates", len(candidates))
    print("  sample losses", np.round(sample_losses, 4).tolist())


## Run the same method across D1-D5

Every rung uses the same GP-UCB Bayesian optimizer. The metric is best observed loss; lower is better.

In [ ]:

bayes_results = []

for index, rung in enumerate(rungs, start=1):
    result = bayesian_minimize(
        rung["objective"],
        rung["bounds"],
        iterations=16,
        initial_count=3,
        grid_size=rung["grid"],
        length_scale=0.45,
        noise=1e-5,
        kappa=1.5,
    )
    bayes_results.append(result)
    print(f"D{index} | best observed loss {result['best_curve'][-1]:.5f} | evals {len(result['losses'])}")


## Results visualization

The left panels show sampled locations on the candidate landscape. The right panel shows best observed loss versus evaluation budget.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
flat_axes = axes.ravel()

for index, result in enumerate(bayes_results):
    ax = flat_axes[index]
    rung = rungs[index]
    path = result["path"]
    if path.shape[1] == 1:
        ax.plot(path[:, 0], np.arange(len(path)), marker="o")
        ax.set_xlabel("x")
        ax.set_ylabel("sample order")
    else:
        ax.scatter(path[:, 0], path[:, 1], c=np.arange(len(path)), cmap="viridis")
        ax.set_xlabel("x0")
        ax.set_ylabel("x1")
    ax.set_title(f"D{index + 1} trajectory")

summary_ax = flat_axes[-1]
for index, result in enumerate(bayes_results, start=1):
    summary_ax.plot(result["best_curve"], marker="o", label=f"D{index}")

summary_ax.set_title("best observed loss vs iteration")
summary_ax.set_xlabel("Bayesian optimization step")
summary_ax.set_ylabel("best loss")
summary_ax.legend()
plt.tight_layout()


## Pitfall on D5: posterior mean alone

Optimizing $\mu(x)$ alone removes the exploration term $\sigma(x)$ and can overcommit to early observations. The fix restores uncertainty-aware UCB and a small noise term.

In [ ]:

d5 = rungs[-1]
mean_only = bayesian_minimize(
    d5["objective"],
    d5["bounds"],
    iterations=16,
    initial_count=3,
    grid_size=d5["grid"],
    length_scale=0.45,
    noise=1e-12,
    kappa=0.0,
)
ucb_fixed = bayesian_minimize(
    d5["objective"],
    d5["bounds"],
    iterations=16,
    initial_count=3,
    grid_size=d5["grid"],
    length_scale=0.45,
    noise=1e-4,
    kappa=2.0,
)

print("mean-only final best", mean_only["best_curve"][-1])
print("uncertainty-aware final best", ucb_fixed["best_curve"][-1])

assert np.isfinite(mean_only["best_curve"][-1])
assert np.isfinite(ucb_fixed["best_curve"][-1])


## Evaluate it + Practice

- Metric: track best observed loss against a no-skill baseline: uniform random candidate evaluations under the same budget.
- Cheap sanity check: posterior mean at observed x=1 must be about 1.000.
- Ablation: set kappa to 0 and compare against UCB on D5.
- Failure signals: the same point is repeatedly sampled, uncertainty collapses incorrectly, or noisy data are interpolated as exact.

### Practice

- Change the RBF length scale on D3 and explain how the acquisition path changes.

- Replace UCB with expected improvement and compare best-loss curves.

- Add duplicate noisy observations and increase the GP noise term.